# T6.2 LLM extraction on Colab Pro+

Rerouted from OSC (DECISIONS 2026-07-04). Same vLLM + Outlines + YaRN engine, on a Colab GPU.
Read `cloud/colab/README.md` + `RUNBOOK.md` first. Prereqs: Pro+ with a **GPU runtime**
(Runtime → Change runtime type → A100/L4), the repo on Drive at the `REPO` path below, and the
two gitignored `chunks.parquet` staged under its `data/`.

Reproducibility rule: a model's κ-audit must score the exact weights that produced its corpus
features — so `--audit-sample` and the corpus run use the same model + engine in the same session.

In [ ]:
# 1. Mount Drive (persistent store: code + the data/ outputs survive VM disconnects)
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
# 2. cd into the repo on Drive + install (idempotent; safe to re-run after a reconnect)
REPO = "/content/drive/MyDrive/ecvol/Earnings_call_project-main"  # <-- edit if yours differs
%cd $REPO
!bash cloud/colab/setup.sh

## Model panel

Run **smoke → κ-gate → corpus** once per model, in panel order, editing `PANEL[i]` below each
time. Each model must clear **κ>0.6** on its own audit before its corpus run — the audit and the
corpus use the same weights (reproducibility rule), so never skip the gate for a panel member.

- **Qwen2.5-7B-Instruct** — the confirmatory candidate; fp16, fits the 40 GB A100 at 65k.
- **Qwen2.5-32B-Instruct-AWQ** — exploratory *does-scale-help* check. Served as **AWQ 4-bit**
  because fp16 32B (~64 GB) won't fit 40 GB; vLLM auto-detects AWQ from the checkpoint (no extra
  flag). At 65k the KV cache is tight on 40 GB — if it OOMs on the longest sections that is the
  GPU-size limit, not a policy change (don't lower `--max-model-len`; re-roll the GPU or skip).
- **Llama-3.1-8B-Instruct** — cross-family check; fp16 fits. Gated → run `huggingface-cli login`
  once (HF token) before this model; Qwen checkpoints are ungated.

72B is dropped on Colab (needs >40 GB even at 4-bit — it was an 80 GB-only OSC option).

In [ ]:
# 3. Config the run — pick the panel model for THIS session (see the panel notes above)
PANEL = [
    "Qwen/Qwen2.5-7B-Instruct",  # confirmatory candidate; fp16 fits 40GB @65k
    "Qwen/Qwen2.5-32B-Instruct-AWQ",  # exploratory scale check; AWQ 4-bit (fp16 32B > 40GB)
    "meta-llama/Llama-3.1-8B-Instruct",  # cross-family check; gated (hf login first)
]
MODEL = PANEL[0]  # <-- set the index (0/1/2) for the model you're running now
CTX = "--max-model-len 65536 --yarn"  # >32k policy: extend (must match audit+corpus)
SLUG = MODEL.replace("/", "__").replace(":", "_")
print(MODEL, "->", f"data/*/llm_features__{SLUG}.parquet")

In [ ]:
# 4. Smoke test — 3 calls, validates the vLLM/YaRN path cheaply
!ecvol featurize llm --dataset fincall --model-id $MODEL \
    --engine vllm $CTX --limit 3 --root data

In [ ]:
# 5. κ-GATE — extract the 50 audit calls (same engine), then score; corpus BLOCKED until PASS
!ecvol featurize llm --dataset fincall --model-id $MODEL \
    --engine vllm $CTX --audit-sample --root data
!ecvol llm-kappa --sheet data/coverage/fincall_llm_labels_rater1.csv \
    --features data/fincall/llm_features__$SLUG.parquet

In [ ]:
# 6. Full corpus — ONLY if the gate passed. Resumes over audit calls; re-run after a disconnect.
!ecvol featurize llm --dataset fincall --model-id $MODEL \
    --engine vllm $CTX --root data
!ecvol featurize llm --dataset maec --model-id $MODEL \
    --engine vllm $CTX --root data